# HW3 Part 2


**Course Code:** 515513

**Group number:** 515513_07

**Student Name:** 呂泰廷 吳瑞傑 郭宗睿

**Student ID:** 112652030 112652027 111652043

# Turn Continuity Classification

**Task**: Binary classification — predict whether a spoken turn is **Complete (1)** or **Incomplete (0)**.

**Metric**: Macro-F1 Score.

| Label | Meaning |
|---|---|
| 1 | **Complete** — semantic intent is finished; system can respond |
| 0 | **Incomplete** — intent is unfinished; system should keep listening |

## 0. Environment Setup & Data Loading

In [2]:
!pip install datasets xgboost imbalanced-learn nltk -q


[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')

# Core ML
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.metrics import classification_report, f1_score
from sklearn.utils import resample
from sklearn.pipeline import Pipeline
from sklearn.metrics import confusion_matrix

# Advanced
import xgboost as xgb
from imblearn.over_sampling import SMOTE

# NLP
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
nltk.download('wordnet', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('omw-1.4', quiet=True)


print("All imports successful!")

All imports successful!


In [4]:
# Load dataset (ensure train.csv and test.csv are in your current directory)
train_path = 'train.csv'
test_path = 'test.csv'

train_df = pd.read_csv(train_path)
public_test_df = pd.read_csv(test_path)

# Basic exploration
print(f"Train size: {len(train_df)}")
print("Label Distribution:\n", train_df['label'].value_counts())
display(train_df.head())

Train size: 1302
Label Distribution:
 label
1    837
0    465
Name: count, dtype: int64


,id,content,label
0,1166,i think we should consider the actually the cl...,1
1,1127,Can you reset my password? My account password...,1
2,1240,im wondering if we need to no wait the test ca...,1
3,1853,"This code needs to be reviewed by tomorrow, do...",0
4,194,"This homework is taking forever, btw the new s...",0


In [5]:
# ── Standardize column names ──────────────────────────────────────

# Based on your files, the text column is named 'content'
# We will normalize it to 'text' for consistency in the pipeline
text_col_mapping = {'content': 'text'}

# Rename 'content' to 'text' if it exists
train_df = train_df.rename(columns=text_col_mapping)
public_test_df = public_test_df.rename(columns=text_col_mapping)

# Ensure 'id' column exists (if not already present)
if 'id' not in train_df.columns:
    train_df['id'] = train_df.index
if 'id' not in public_test_df.columns:
    public_test_df['id'] = public_test_df.index

# Ensure text columns are strings
train_df['text'] = train_df['text'].fillna('').astype(str)
public_test_df['text'] = public_test_df['text'].fillna('').astype(str)

# ── Label Analysis (Training Set Only) ────────────────────────────

print("Standardization complete.")
print(f"Final columns: {train_df.columns.tolist()}")

if 'label' in train_df.columns:
    counts = train_df['label'].value_counts()
    print("\nLabel distribution (train):")
    print(counts)

    if 0 in counts and 1 in counts:
        ratio = counts[0] / counts[1]
        print(f"\nImbalance ratio (0/1): {ratio:.2f}")
    else:
        print("\nNote: Only one class found in 'label' column.")
else:
    print("\nWarning: 'label' column not found in train_df.")

Standardization complete.
Final columns: ['id', 'text', 'label']

Label distribution (train):
label
1    837
0    465
Name: count, dtype: int64

Imbalance ratio (0/1): 0.56


## Part 1: Data Balancing

You must implement and compare two methods:
1. **Basic (Required)**: Random Over-sampling.
2. **Advanced (Choose 1+)**: EDA, Back-translation, SMOTE, or Cost-Sensitive Learning.

We choose **Cost-Sensitive Learning** as the advanced method. It re-weights the loss function instead of duplicating rows, which mathematically achieves balanced training without introducing duplicate samples that can mildly over-fit.

In [ ]:
# =================================================================
# Phase 1 - 資料平衡
# =================================================================
# 必做：Random Over-sampling -> 把少數類複製到 1:1
# 進階：Cost-Sensitive Learning -> 不動資料，改在 SVM 加 class_weight='balanced'
# 兩個方法用「同一個模型」(TF-IDF + Linear SVM) 比較，才能公平。

def perform_balancing(df, method='random'):
    if method == 'random':
        # --- Step 1: 按 label 分成多數類 / 少數類 ---
        # label 1 (Complete)  = 多數類 (837 筆)
        # label 0 (Incomplete)= 少數類 (465 筆)
        df_majority = df[df['label'] == 1]
        df_minority = df[df['label'] == 0]

        # --- Step 2: 把少數類「有放回」抽樣到跟多數類一樣多 ---
        df_minority_upsampled = resample(
            df_minority,
            replace=True,                   # 有放回抽樣
            n_samples=len(df_majority),     # 抽到跟多數類一樣多 -> 1:1
            random_state=42,                # 固定隨機種子，結果可重現
        )

        # --- Step 3: 合併 + 打散 ---
        df_balanced = pd.concat([df_majority, df_minority_upsampled])
        df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)
        return df_balanced

    elif method == 'advanced':
        # --- Cost-Sensitive Learning：不動資料 ---
        # 等下訓練時在 SVM 設 class_weight='balanced'，
        # 自動把每個樣本的 loss 乘以 n_samples / (n_classes * class_count)
        # 效果等同完美平衡的過採樣，但「不複製任何一列」-> 不會造成 variance 膨脹
        return df.copy().reset_index(drop=True)

# Quick smoke-test：確認兩種方法的輸出符合預期
sample_balanced = perform_balancing(train_df, method='random')
print("After Random Over-sampling (兩類 1:1):")
print(sample_balanced['label'].value_counts())

sample_adv = perform_balancing(train_df, method='advanced')
print("\nAfter Cost-Sensitive (資料不變，靠 class_weight 平衡):")
print(sample_adv['label'].value_counts())


## Part 2: Baseline Classifier (TF-IDF + SVM)

Establish a baseline. Use Macro-F1 as your primary metric.

In [ ]:
# =================================================================
# Phase 3 - 文字前處理：在句尾「明文標記」結束點訊號
# =================================================================
# EDA 觀察：label 0 (Incomplete) 結尾幾乎都是虛詞 (the/to/of/about/could/if)，
#           label 1 (Complete)   結尾幾乎都是內容詞 (it/please/today/town)。
# 但 unigram TF-IDF 會把這個訊號攤平 -> the 在句中也常出現，模型很難學到
# 「結尾是 the -> Incomplete」這條規則。
# 解法：在每句尾巴另外附加 5 個 marker token，把訊號直接寫進特徵。

# --- 7 大類「結尾詞」字典（用來判斷句尾屬於哪一類）---
FILLER = {'um','uh','er','like','maybe','actually','wait','well','okay','ok','so','just'}
DET    = {'the','a','an','my','your','his','her','its','our','their','this','that','these','those','some','any','no','every'}  # 限定詞
PREP   = {'of','to','in','on','at','with','for','from','about','into','over','under','through','during','before','after','between','as','by'}  # 介詞
CONJ   = {'and','but','or','because','if','when','while','though','although','since','unless','that','whether','so','than'}  # 連接詞
MODAL  = {'can','could','will','would','shall','should','may','might','must','do','does','did','have','has','had','is','was','were','am','are','be','been','being'}  # 助動詞 / be 動詞
PRON   = {'i','you','he','she','it','we','they','me','him','us','them','myself','yourself','himself','herself','itself','ourselves','themselves'}  # 代名詞

def _cat(w):
    """把一個字對應到上面 7 大類；不屬於任何一類就當作內容詞 (CONTENT)。"""
    if w in FILLER: return 'FILLER'
    if w in DET:    return 'DET'
    if w in PREP:   return 'PREP'
    if w in CONJ:   return 'CONJ'
    if w in MODAL:  return 'AUX'
    if w in PRON:   return 'PRON'
    return 'CONTENT'

_TOKRE = re.compile(r"[A-Za-z']+")  # 只抽英文字

def preprocess_text(text):
    """在原句後面追加 5 個結尾 marker token，讓 SVM 直接看到結尾訊號。"""
    if not isinstance(text, str):
        return ''
    t = text.lower().strip()
    toks = _TOKRE.findall(t)            # tokenize
    if not toks:
        return t
    end1 = toks[-1]                                                     # 最後一個字
    end2 = '_'.join(toks[-2:])  if len(toks) >= 2 else end1             # 倒數兩個字 (bigram)
    end3 = '_'.join(toks[-3:])  if len(toks) >= 3 else end2             # 倒數三個字 (trigram)
    cat1 = _cat(end1)                                                   # 最後字屬於哪一類
    cat2 = _cat(toks[-2]) if len(toks) >= 2 else cat1                   # 倒數第二字屬於哪一類
    # 附加 5 個 marker：__END1/2/3__ + __CAT1__ + __CAT12__
    return f'{t} __END1_{end1}__ __END2_{end2}__ __END3_{end3}__ __CAT1_{cat1}__ __CAT12_{cat2}_{cat1}__'

# 套用到 train / test 兩邊 (用法相同，避免訓練 / 推論不一致)
train_df['text_clean']       = train_df['text'].apply(preprocess_text)
public_test_df['text_clean'] = public_test_df['text'].apply(preprocess_text)

# 印出範例：對照原始句子 vs. 加上 marker 之後的句子
print('原始 :', train_df['text'].iloc[0])
print('增強 :', train_df['text_clean'].iloc[0])


In [ ]:
# =================================================================
# Phase 2 - 模型工廠
# =================================================================
# 提供兩種模型給後面的實驗呼叫：
#   'svm'      = Linear SVM（必做 baseline；class_weight 由外部決定）
#   'advanced' = XGBoost   （選做的進階模型對照組）
def get_model(model_type='svm', class_weight=None):
    if model_type == 'svm':
        # Linear SVM：適合 TF-IDF 這種高維稀疏特徵
        # class_weight='balanced' 時就是 Cost-Sensitive Learning
        return SVC(kernel='linear', C=1.0, class_weight=class_weight,
                   probability=True, random_state=42)

    elif model_type == 'advanced':
        # XGBoost：tree-based ensemble，當對照用
        return xgb.XGBClassifier(
            n_estimators=400,        # 樹的數量
            max_depth=6,             # 每棵樹的最大深度
            learning_rate=0.1,
            subsample=0.9,
            colsample_bytree=0.9,
            eval_metric='logloss',
            random_state=42,
            n_jobs=-1,
        )


In [9]:
# --- Step 1: Internal Validation Setup ---
train_data, val_data = train_test_split(
    train_df, test_size=0.2, random_state=42, stratify=train_df['label'])


print(f"Train subset : {len(train_data)} samples")
print(f"Val   subset : {len(val_data)} samples")
print("Train label distribution:")
print(train_data['label'].value_counts())

Train subset : 1041 samples
Val   subset : 261 samples
Train label distribution:
label
1    669
0    372
Name: count, dtype: int64


In [ ]:
# =================================================================
# Baseline 實驗：TF-IDF unigram + Linear SVM + Random Over-sampling
# =================================================================
# 流程：1. 平衡訓練資料 -> 2. TF-IDF 抽特徵 -> 3. 訓練 SVM -> 4. Macro-F1 評估

print("Running Baseline Experiment...")

MODEL_TYPE     = 'svm'      # 用必做的 Linear SVM
BALANCE_METHOD = 'random'   # 用必做的 Random Over-sampling

# --- 1. 平衡資料 (1:1) ---
balanced_train_data = perform_balancing(train_data.copy(), method=BALANCE_METHOD)
print(f"Balanced train size: {len(balanced_train_data)}  |  label dist:")
print(balanced_train_data['label'].value_counts())

# --- 2. TF-IDF 抽特徵 ---
# ngram_range=(1,1) -> 只用 unigram；min_df=2 -> 出現少於 2 次的字直接丟掉
vectorizer = TfidfVectorizer(ngram_range=(1, 1), min_df=2, stop_words=None)
X_train = vectorizer.fit_transform(balanced_train_data['text_clean'])  # 注意只 fit 訓練集
y_train = balanced_train_data['label']

# --- 3. 訓練 SVM ---
clf = get_model(MODEL_TYPE)
clf.fit(X_train, y_train)
print(f"Baseline model ({MODEL_TYPE}) trained on {X_train.shape[0]} balanced samples.")

# --- 4. 在 validation set 上算 Macro-F1 ---
X_val = vectorizer.transform(val_data['text_clean'])     # 用同一個 vectorizer transform
y_val = val_data['label']
y_pred = clf.predict(X_val)
macro_f1 = f1_score(y_val, y_pred, average='macro')

print(f"=== Baseline: {MODEL_TYPE} | Balancing: {BALANCE_METHOD} ===")
print(classification_report(y_val, y_pred, target_names=['Incomplete(0)', 'Complete(1)']))
print(f"Macro-F1 Score: {macro_f1:.4f}")


In [11]:
# Compare Random Over-sampling vs Cost-Sensitive Learning (same SVM model)

results_bal = {}
for bal in ['random', 'advanced']:
    bal_tr = perform_balancing(train_data.copy(), method=bal)
    cw = 'balanced' if bal == 'advanced' else None
    v = TfidfVectorizer(ngram_range=(1, 1), min_df=2)
    Xtr = v.fit_transform(bal_tr['text_clean'])
    Xva = v.transform(val_data['text_clean'])
    m = get_model('svm', class_weight=cw)
    m.fit(Xtr, bal_tr['label'])
    p = m.predict(Xva)
    score = f1_score(val_data['label'], p, average='macro')
    results_bal[bal] = score
    label = 'Random Over-sampling' if bal == 'random' else 'Cost-Sensitive (advanced)'
    print(f'{label:32s}  macro-F1 = {score:.4f}')

print('\nBest balancing method:', max(results_bal, key=results_bal.get))

Random Over-sampling              macro-F1 = 0.8565
Cost-Sensitive (advanced)         macro-F1 = 0.8774

Best balancing method: advanced


## Part 3: Enhancement with K-Fold (Optional)

Improve your results with implement  K-FOLD Cross Validtaion if needed.

In [ ]:
# =================================================================
# Phase 3 - Stratified K-Fold 交叉驗證
# =================================================================
# 為什麼要做 K-Fold？單一 80/20 切分可能只是「好運」的結果；
# 5-fold 把 train 切成 5 份，每份輪流當 validation，
# 平均分數 + 標準差才能反映模型真實的泛化能力。
#
# Stratified 表示每個 fold 都保留原本的類別比例 (label 0 : 1 接近 36% : 64%)，
# 在不平衡資料上很重要。

def run_kfold_cv(df, model_type='svm', balance_method='random', n_splits=5,
                 use_custom_preprocessing=True, ngram_range=(1, 2),
                 sublinear_tf=True, min_df=2, C=1.0):
    """跑 Stratified K-Fold CV，回傳每個 fold 的 macro-F1。"""
    # use_custom_preprocessing=True 表示用 endpoint augment 過的 text_clean
    text_col = 'text_clean' if use_custom_preprocessing else 'text'
    # balance_method='advanced' 時，模型用 class_weight='balanced'
    cw = 'balanced' if balance_method == 'advanced' else None

    X_all = df[text_col].values
    y_all = df['label'].values
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

    fold_scores = []
    for fold, (tri, vai) in enumerate(skf.split(X_all, y_all), 1):
        # 切出本 fold 的 train / val
        tr_df_fold = df.iloc[tri].reset_index(drop=True)
        va_df_fold = df.iloc[vai].reset_index(drop=True)

        # 只有 random 平衡需要實際改資料；advanced 靠模型權重
        if balance_method == 'random':
            tr_df_fold = perform_balancing(tr_df_fold, method='random')

        # TF-IDF 只在訓練 fold 上 fit -> 避免 validation 字彙洩漏
        v = TfidfVectorizer(ngram_range=ngram_range, min_df=min_df,
                            sublinear_tf=sublinear_tf)
        Xtr = v.fit_transform(tr_df_fold[text_col])
        Xva = v.transform(va_df_fold[text_col])

        # 模型訓練 + 預測 + 算 macro-F1
        if model_type == 'svm':
            m = SVC(kernel='linear', C=C, class_weight=cw, random_state=42)
        else:
            m = get_model('advanced')
        m.fit(Xtr, tr_df_fold['label'])
        p = m.predict(Xva)
        sc = f1_score(va_df_fold['label'], p, average='macro')
        fold_scores.append(sc)
        print(f'  fold {fold}/{n_splits}  macro-F1 = {sc:.4f}')
    print(f'>> mean = {np.mean(fold_scores):.4f}  std = {np.std(fold_scores):.4f}')
    return fold_scores

# --- 三組設定一次跑完，互相比較 ---

# (1) Baseline：原始文字 + unigram + Random Over-sampling
print('--- Baseline: SVM unigram + random over-sampling (原始文字) ---')
base_scores = run_kfold_cv(train_df, model_type='svm', balance_method='random',
                           use_custom_preprocessing=False, ngram_range=(1, 1),
                           sublinear_tf=False)

# (2) Advanced：endpoint augment + (1,3)-gram + sublinear + Cost-Sensitive
print('\n--- Advanced: SVM (1,3)-gram + sublinear + cost-sensitive + endpoint augment ---')
adv_scores = run_kfold_cv(train_df, model_type='svm', balance_method='advanced',
                          use_custom_preprocessing=True, ngram_range=(1, 3),
                          sublinear_tf=True, C=0.5)

# (3) Reference：XGBoost 對照組（看看換模型有沒有比較強）
print('\n--- Reference: XGBoost on TF-IDF (1,2)-gram + sublinear ---')
xgb_scores = run_kfold_cv(train_df, model_type='advanced', balance_method='random',
                          use_custom_preprocessing=True, ngram_range=(1, 2),
                          sublinear_tf=True)


## Part 4: Final Submission

Train on the full dataset using your best found configuration and generate `submission.csv`.

In [13]:
# Train final model on the entire training set using the BEST config found in CV
# Best config: SVM linear, C=0.5, class_weight='balanced',
#              TF-IDF ngram=(1,3), sublinear_tf=True, min_df=2,
#              text = endpoint-aware augmented preprocess_text output.

BEST_BALANCE = 'advanced'        # cost-sensitive learning
final_balanced_train_df = perform_balancing(train_df.copy(), method=BEST_BALANCE)
print(f"Full training size: {len(final_balanced_train_df)}")
print(final_balanced_train_df['label'].value_counts())

final_vectorizer = TfidfVectorizer(ngram_range=(1, 3), min_df=2, sublinear_tf=True)
X_full_train = final_vectorizer.fit_transform(final_balanced_train_df['text_clean'])
y_full_train = final_balanced_train_df['label']

final_model = SVC(kernel='linear', C=0.5, class_weight='balanced', random_state=42)
final_model.fit(X_full_train, y_full_train)
print(f"Final model trained on {X_full_train.shape[0]} samples (augmented preprocess_text).")


Full training size: 1302
label
1    837
0    465
Name: count, dtype: int64
Final model trained on 1302 samples (augmented preprocess_text).


In [14]:
# -----------------------------------------------------------------
# PHASE 4: Kaggle Submission
# -----------------------------------------------------------------

def generate_kaggle_submission(model, vectorizer, test_df, output_name='submission.csv'):
    """
    Generates the final CSV. test_df['text_clean'] is processed the same way as training data.
    """
    text_col = 'text_clean' if 'text_clean' in test_df.columns else 'text'
    X_test = vectorizer.transform(test_df[text_col])
    test_predictions = model.predict(X_test).astype(int)

    submission = pd.DataFrame({'id': test_df['id'].astype(int), 'label': test_predictions})
    submission.to_csv(output_name, index=False)

    print(f"Saved → {output_name}")
    print("Prediction distribution:")
    print(submission['label'].value_counts())
    display(submission.head(10))

    print(f"Success: {output_name} ready for Kaggle upload.")

# Final Step:
generate_kaggle_submission(final_model, final_vectorizer, public_test_df)

Saved → submission.csv
Prediction distribution:
label
1    304
0    196
Name: count, dtype: int64


,id,label
0,1608,0
1,334,1
2,1779,1
3,1015,1
4,1790,1
5,1635,1
6,358,0
7,1574,1
8,700,1
9,284,1


Success: submission.csv ready for Kaggle upload.


In [15]:
# Download in Colab
try:
    from google.colab import files
    files.download('submission.csv')
    print("Download started!")
except ImportError:
    print("Not running in Colab — file saved locally as submission.csv")

Not running in Colab — file saved locally as submission.csv
